In [1]:
#r "nuget: ScottPlot, 5.0.39"
#r "C:\Users\Para8e11um\projects\practice2026\task17\bin\Debug\net9.0\task17.dll"
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using Microsoft.DotNet.Interactive;
using task17;

record ExecutionEvent(int TaskId, int Iteration, double TimeMs, bool IsSchedulerUsed);
var events = new List<ExecutionEvent>();
var stopwatch = new Stopwatch();
int iterations = 5;
int workDelayMs = 50;

Installed Packages ScottPlot, 5.0.39

Loading extensions from `C:\Users\Para8e11um\.nuget\packages\skiasharp\2.88.8\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

In [ ]:
class MonopolyCommand : ICommand
{
    private int _id;
    private int _iterations;
    private int _workDelayMs;
    private List<ExecutionEvent> _events;
    private Stopwatch _stopwatch;
    public MonopolyCommand(int id,int iterations, int workDelayMs, List<ExecutionEvent> events, Stopwatch stopwatch){
        _id = id;
        _iterations = iterations; 
        _workDelayMs = workDelayMs;
        _events = events;  
        _stopwatch = stopwatch;
    }
    public void Execute()
    {
        for (int i = 1; i <= _iterations; i++)
            {
                lock (_events) _events.Add(new ExecutionEvent(_id, i, _stopwatch.ElapsedMilliseconds, false));
                Thread.Sleep(_workDelayMs);
            }
    }
}

class YieldingCommand : ICommand
{
    private int _id, _current = 0;
    private IScheduler _scheduler;
    private int _iterations;
    private int _workDelayMs;
    private List<ExecutionEvent> _events;
    private Stopwatch _stopwatch;
    public YieldingCommand(int id, IScheduler scheduler, int iterations, int workDelayMs, List<ExecutionEvent> events, Stopwatch stopwatch)
    {
        _id = id;
        _scheduler = scheduler;
        _iterations = iterations;
        _workDelayMs = workDelayMs;
        _events = events;
        _stopwatch = stopwatch;
    }
    public void Execute()
    {
        _current++;
        lock (_events) _events.Add(new ExecutionEvent(_id, _current, _stopwatch.ElapsedMilliseconds, true));
        Thread.Sleep(_workDelayMs);
        if (_current < _iterations)
        {
            _scheduler.Add(this);
        }
    }
}

var scheduler = new RoundRobinScheduler();
var server = new ServerThread(scheduler);
server.Start();
stopwatch.Start();

Console.WriteLine("Запуск задач БЕЗ планировщика");
server.Enqueue(new MonopolyCommand(1, iterations, workDelayMs, events, stopwatch));
server.Enqueue(new MonopolyCommand(2, iterations, workDelayMs, events, stopwatch));
server.Enqueue(new MonopolyCommand(3, iterations, workDelayMs, events, stopwatch));

Thread.Sleep(1000);

double phaseTwoStartTime = stopwatch.ElapsedMilliseconds + 200;
Thread.Sleep(300);

Console.WriteLine("Запуск задач С планировщиком (Round-Robin)");
server.Enqueue(new YieldingCommand(1, scheduler, iterations, workDelayMs, events, stopwatch));
server.Enqueue(new YieldingCommand(2, scheduler, iterations, workDelayMs, events, stopwatch));
server.Enqueue(new YieldingCommand(3, scheduler, iterations, workDelayMs, events, stopwatch));
Thread.Sleep(1000);

server.Enqueue(new HardStopCommand(server));
stopwatch.Stop();
Console.WriteLine("Сервер остановлен");

var plt = new Plot();
plt.Title("Сравнение режимов выполнения на ServerThread");
plt.YLabel("Номер задачи");
plt.XLabel("Время выполнения (мс)");
double[] yTicks = { 1, 2, 3 };
string[] yLabels = { "Задача 1", "Задача 2", "Задача 3" };
plt.Axes.Left.SetTicks(yTicks, yLabels);
plt.Axes.Left.Max = 3.5;
plt.Axes.Left.Min = 0.5;
foreach (var t in new[] { 1, 2, 3 })
{
    var evsSeq = events.Where(e => !e.IsSchedulerUsed && e.TaskId == t).ToList();
    if (evsSeq.Any()) {
        var spSeq = plt.Add.ScatterPoints(evsSeq.Select(e => e.TimeMs).ToArray(), evsSeq.Select(e => (double)t).ToArray());
        spSeq.Color = Colors.Red; spSeq.MarkerSize = 10;
    }
    var evsSched = events.Where(e => e.IsSchedulerUsed && e.TaskId == t).ToList();
    if (evsSched.Any()) {
        var spSched = plt.Add.ScatterPoints(evsSched.Select(e => e.TimeMs).ToArray(), evsSched.Select(e => (double)t).ToArray());
        spSched.Color = Colors.Blue; spSched.MarkerSize = 10;
    }
}
var line = plt.Add.VerticalLine(phaseTwoStartTime);
line.LineWidth = 2; line.Color = Colors.Black; line.LinePattern = LinePattern.Dashed;
string imgPath = "real_scheduler_comparison.png";
plt.SavePng(imgPath, 800, 300);
// ==========================================
// 4. Генерация отчета (report.txt)
// ==========================================
string reportPath = "report.txt";
using (var writer = new StreamWriter(reportPath))
{
    writer.WriteLine("Замеры работы ServerThread и RoundRobinScheduler");
    writer.WriteLine("\n[БЕЗ ПЛАНИРОВЩИКА] Время старта (задержка отклика):");
    double t1SeqStart = events.First(e => !e.IsSchedulerUsed && e.TaskId == 1).TimeMs;
    foreach (var t in new[] { 1, 2, 3 })
    {
        var start = events.First(e => !e.IsSchedulerUsed && e.TaskId == t).TimeMs;
        writer.WriteLine($"  - Задача {t}: задержка от старта первой задачи = {start - t1SeqStart} мс");
    }
    writer.WriteLine("\n[С ПЛАНИРОВЩИКОМ] Время старта (задержка отклика):");
    double t1SchedStart = events.First(e => e.IsSchedulerUsed && e.TaskId == 1).TimeMs;
    foreach (var t in new[] { 1, 2, 3 })
    {
        var start = events.First(e => e.IsSchedulerUsed && e.TaskId == t).TimeMs;
        writer.WriteLine($"  - Задача {t}: задержка от старта первой задачи = {start - t1SchedStart} мс");
    }
}

HTML($"<img src='{imgPath}' alt='График сравнения' />").Display();
Console.WriteLine(File.ReadAllText(reportPath));

Запуск задач БЕЗ планировщика
Запуск задач С планировщиком (Round-Robin)
Сервер остановлен


Замеры работы ServerThread и RoundRobinScheduler

[БЕЗ ПЛАНИРОВЩИКА] Время старта (задержка отклика):
  - Задача 1: задержка от старта первой задачи = 0 мс
  - Задача 2: задержка от старта первой задачи = 309 мс
  - Задача 3: задержка от старта первой задачи = 620 мс

[С ПЛАНИРОВЩИКОМ] Время старта (задержка отклика):
  - Задача 1: задержка от старта первой задачи = 0 мс
  - Задача 2: задержка от старта первой задачи = 61 мс
  - Задача 3: задержка от старта первой задачи = 123 мс

